# **Assignment 1 - Crew Scheduling Problem**


```
The format of these data files is:  
   number of rows, number of columns (n)  
   for each column j (j=1,...,n) in turn:  
      column cost, number of rows covered by j, list of the rows covered by j  
```

### **Problem Statement:**
 + Select a subset of columns to minimize cost while following the constraint that each row must be covered EXACTLY ONCE by the subset.
 + From googling, it looks like the true optimals are: `sppnw41: 11307` `sppnw42: 7656` `sppnw43: 8904`

In [5]:
# Variable Setup

import numpy as np
f_name ='sppnw43'
file_path = f'data/{f_name}.txt'
with open(file_path, 'r') as f:
    data = f.readlines()

if f_name == 'sppnw41': optimal = 11307
elif f_name == 'sppnw42': optimal = 7656
elif f_name == 'sppnw43': optimal = 8904
else: optimal = 0

data = [[int(x) for x in d.strip().split()] for d in data]

N_ROWS,N_COLS = data[0]

COL_COSTS = np.array([d[0] for d in data[1:]])
COL_ROWS = np.array([np.isin(np.arange(1,N_ROWS+1), d[2:]) for d in data[1:]])

print(f"Num Rows: {N_ROWS} | Num Columns: {N_COLS}")
print(f"Avg Rows per Column: {np.mean([len(rows[rows]) for rows in COL_ROWS]):.2f}")

Num Rows: 18 | Num Columns: 1072
Avg Rows per Column: 4.53


In [10]:
# Getting some idea of scale

total_cost = np.sum(COL_COSTS); avg_cost = np.mean(COL_COSTS)
min_cost = np.min(COL_COSTS); max_cost = np.max(COL_COSTS)
std_cost = np.std(COL_COSTS)

print(f"Total cost: {total_cost} | Avg Cost: {avg_cost:.0f} | STD of Cost: {std_cost:.0f}")
print(f"Min Cost: {min_cost} | Max Cost: {max_cost}")

Total cost: 3402908 | Avg Cost: 3174 | STD of Cost: 1417
Min Cost: 110 | Max Cost: 7130


In [7]:
# Improved BGA


# ---- Binary Representation

#   Individuals are defined by a binary string COLUMNS long
INDIVIDUAL_SHAPE = (N_COLS)

POPULATION_SIZE = 128

NUM_PARENTS = int(POPULATION_SIZE * 1/8)
NUM_CHILDREN = 128
NUM_ELITES = int(POPULATION_SIZE * 0.01)

POPULATION_SHAPE = (POPULATION_SIZE, INDIVIDUAL_SHAPE)
# --------------



#   Mutation and Initialization Parameters
MUTATION_RATE = 4.0 / N_COLS #N_COLS # want around 2-6 columns changed per mutation




# ---- Requirement 3.1 - Initialization Algorithm
def init_individual() -> np.ndarray:
    individual = np.zeros(INDIVIDUAL_SHAPE, dtype=bool)
    rows = np.arange(N_ROWS)

    while rows.size != 0:
        row_sums = np.sum(COL_ROWS[individual], axis=0)
        r = np.random.choice(rows)

        candidates = np.where(COL_ROWS[:, r])[0]
        safe = candidates[np.all((COL_ROWS[candidates] == 0) | (row_sums == 0), axis=1)]

        if safe.size > 0:
            col = np.random.choice(safe)
            individual[col] = True
            cov_rows = np.where(COL_ROWS[col])[0]
            rows = np.setdiff1d(rows, cov_rows)
        else:
            rows = np.setdiff1d(rows, [r])

    return individual

def initialization(pop_size):
    return np.array([init_individual() for i in range(pop_size)], dtype=bool)
# ----



# ---- Requirement 3.2 - Stochastic Ranking
def stochastic_ranking(p: np.ndarray, N: int=10, pf: float=0.45) -> np.ndarray:
    I = np.arange(p.shape[0])

    costs,penalties = fitness(p)
    for i in range(N):
        has_swapped = False

        for j in range(p.shape[0] - 1):
            a,b = I[j], I[j+1]

            u = np.random.rand()

            if (penalties[a] == 0 and penalties[b] == 0) or u < pf:
                if costs[a] > costs[b]:
                    I[j], I[j+1] = I[j+1], I[j]
                    has_swapped = True
            else:
                if penalties[a] > penalties[b]:
                    I[j], I[j+1] = I[j+1], I[j]
                    has_swapped = True

        if not has_swapped:
            break

    return I
# ----



# ---- Requirement 3.3 - Heuristic Improvement Operator
coverage = np.sum(COL_ROWS, axis=1)
COL_COST_PER_ROW = np.divide(COL_COSTS, coverage, where=coverage > 0)
COL_COST_PER_ROW[coverage == 0] = np.inf

def heuristic_improvement_operator(solution: np.ndarray) -> np.ndarray:
    w = np.sum(COL_ROWS[solution], axis=0)

    t = np.where(solution)[0]
    while t.size:
        col = np.random.choice(t)
        t = t[t != col]

        if np.any(w[COL_ROWS[col]] >= 2):
            solution[col] = False
            w = np.sum(COL_ROWS[solution], axis=0)

    u = np.where(w == 0)[0]
    v = u.copy()
    while v.size > 0:
        r = np.random.choice(v)
        v = v[v != r]

        candidates = np.where(COL_ROWS[:, r])[0]
        not_in_u = np.setdiff1d(np.arange(N_ROWS), u)
        safe_costs = [(c,COL_COST_PER_ROW[c]) for c in candidates if np.all(~COL_ROWS[c, not_in_u])] # All the rows not_in_u must be 0.

        best = min(safe_costs, key=lambda x: x[1])[0] if safe_costs else None

        if best is not None:
            solution[best] = True
            rows = np.where(COL_ROWS[best,:])[0]
            w[rows] += 1
            u = np.setdiff1d(u, rows)
            v = np.setdiff1d(v, rows)
    
    return solution

def heuristic(p: np.ndarray) -> np.ndarray:
    return np.array([heuristic_improvement_operator(x.copy()) for x in p])
# ----



# ---- Fitness -> Returns cost,penalty tuple
def fitness(p: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    row_sums = p.astype(int) @ COL_ROWS
    costs = p @ COL_COSTS

    zero_pen = (row_sums < 1).sum(axis=-1)
    overlap_pen = np.maximum(row_sums - 1, 0).sum(axis=-1)

    penalties = zero_pen + overlap_pen

    return costs,penalties
# ----





def selection(p, pos, num_parents):
    N = p.shape[0]
    parents = []

    for _ in range(num_parents):
        i, j = np.random.randint(0, N, 2)
        winner = i if pos[i] < pos[j] else j
        parents.append(p[winner])

    return np.array(parents)

def mutation(p, pm): # based on   "Lecture5_EvolutionaryAlgorithms-6.pdf" Slide 20/25, using fixed pm
    bit_flips = np.random.rand(*p.shape) < pm
    p[bit_flips] = ~p[bit_flips]
    return p

def crossover(parents, num_children): # based on   "Lecture5_EvolutionaryAlgorithms-6.pdf" Slide 21/25, using uniform crossover
    children = []

    def parent_crossover(x1, x2):
        mask = np.random.rand(INDIVIDUAL_SHAPE) < 0.5
        c1 = x1.copy()
        c2 = x2.copy()
        c1[mask] = x2[mask]
        c2[mask] = x1[mask]
        return c1, c2

    while len(children) < num_children:
        i, j = np.random.choice(len(parents), 2, replace=False)
        c1, c2 = parent_crossover(parents[i], parents[j])

        children.append(c1)

        if len(children) < num_children:
            children.append(c2)

    return np.array(children)



def diversity_hamming_mean(p: np.ndarray) -> float:
    x = p.astype(np.int8, copy=False)
    pop, n = x.shape
    ones = x.sum(axis=0)              # (n,)
    zeros = pop - ones
    return float((2 * ones * zeros).sum() / (pop * (pop - 1)))



def improved_binary_genetic_algorithm(p0: np.ndarray, max_iter=10000):
    if p0.shape != POPULATION_SHAPE:
        raise ValueError("Initialisation population shape is invalid.") 
    
    p = p0; gen = 0

    p_rank = stochastic_ranking(p)

    while gen < max_iter:
        p_pos = np.argsort(p_rank)

        p_parent = selection(p, p_pos, NUM_PARENTS)
        p_new = crossover(p_parent, NUM_CHILDREN)
        p_new = mutation(p_new, pm=MUTATION_RATE) 
        p_new = heuristic(p_new)

        p_combined = np.vstack([p,p_new])
        p_combined_rank = stochastic_ranking(p_combined)

        elite_idx = p_rank[:NUM_ELITES]
        remaining = POPULATION_SIZE - NUM_ELITES

        rem_rank_idx = p_combined_rank[~np.isin(p_combined_rank, elite_idx)]
        selected_idx = rem_rank_idx[:remaining]

        p = np.vstack([p[elite_idx], p_combined[selected_idx]])

        p_rank = stochastic_ranking(p)

        # Tracking best & logging
        best_i = p_rank[0]
        best = p[best_i]; 
        best_c, best_pen = fitness(best)

        if gen % 20 == 0:
            #feas_percent = len([p for p in p_feas if p])/len(p_feas)
            diversity = diversity_hamming_mean(p)
            #pop_child_proportion = (len(selected_idx[selected_idx >= len(p_fit)]) / (len(selected_idx) + len(elite_idx)))
            #avg_fit = np.mean(p_fit)
            #print(f"Cost: {c:.0f} | Fit: {fit_best:.0f} | {'X' if c != fit_best else 'Y'} | Gen: {gen} | Feasible %: {feas_percent:.3f} | Diversity: {diversity:.2f} | Pop Child Proportion: {pop_child_proportion:.3f} | Avg Fit: {avg_fit:.0f}")#Penalty Weights: {zero_w:.0f},{overlap_w:.0f} ")
            print(f"Cost: {best_c} | Penalty: {best_pen} | Gen: {gen} | Diversity: {diversity:.2f} ")
        # ----
        
        gen += 1

        if best_c <= optimal and best_pen == 0:
            return best
        if diversity < 1:
            return best

    p_cost, p_pen = fitness(p)

    if np.any(p_pen == 0):
        feas_idx = np.where(p_pen == 0)[0]
        final_i = feas_idx[np.argmin(p_cost[feas_idx])]
    else:
        final_i = np.argmin(p_pen * 10**12 + p_cost)

    return p[final_i]


p0 = initialization(POPULATION_SIZE)
best = improved_binary_genetic_algorithm(p0=p0, max_iter=1000)

best_c,best_pen = fitness(best)

row_sums = np.sum(COL_ROWS[best], axis=0)

print(f"\nFinal number of columns: {best[best].size} | Overlaps: {row_sums[row_sums > 1].size} | Uncovered: {row_sums[row_sums < 1].size}")
print(f"Row Sums: {row_sums}")
print(f"Final Cost: {best_c} | Feasible: {'YES' if best_pen == 0 else 'NO'} | Columns used:", np.where(best)[0])


C:\Users\harry\AppData\Local\Temp\ipykernel_17684\3562222518.py:86: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  COL_COST_PER_ROW = np.divide(COL_COSTS, coverage, where=coverage > 0)


Cost: 15698 | Penalty: 0 | Gen: 0 | Diversity: 10.48 


KeyboardInterrupt: 

In [8]:
results = []

for x in range(30):
    p0 = initialization(POPULATION_SIZE)
    best = improved_binary_genetic_algorithm(p0=p0, max_iter=10000)
    best_c,best_pen = fitness(best)
    results.append((best,best_c,best_pen))


import csv
with open(f"{f_name}_improved_bga_results.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(results)

Cost: 14462 | Penalty: 0 | Gen: 0 | Diversity: 10.66 
Cost: 11098 | Penalty: 0 | Gen: 20 | Diversity: 10.81 
Cost: 11098 | Penalty: 0 | Gen: 40 | Diversity: 10.80 
Cost: 10862 | Penalty: 0 | Gen: 60 | Diversity: 10.68 
Cost: 10752 | Penalty: 0 | Gen: 80 | Diversity: 10.56 
Cost: 10476 | Penalty: 0 | Gen: 100 | Diversity: 10.49 
Cost: 10476 | Penalty: 0 | Gen: 120 | Diversity: 10.55 
Cost: 10042 | Penalty: 0 | Gen: 140 | Diversity: 10.52 
Cost: 10042 | Penalty: 0 | Gen: 160 | Diversity: 10.48 
Cost: 9856 | Penalty: 0 | Gen: 180 | Diversity: 10.38 
Cost: 9706 | Penalty: 0 | Gen: 200 | Diversity: 10.21 
Cost: 9706 | Penalty: 0 | Gen: 220 | Diversity: 10.07 
Cost: 9618 | Penalty: 0 | Gen: 240 | Diversity: 9.94 
Cost: 9618 | Penalty: 0 | Gen: 260 | Diversity: 9.77 
Cost: 9618 | Penalty: 0 | Gen: 280 | Diversity: 9.54 
Cost: 9470 | Penalty: 0 | Gen: 300 | Diversity: 9.44 
Cost: 9290 | Penalty: 0 | Gen: 320 | Diversity: 9.31 
Cost: 9290 | Penalty: 0 | Gen: 340 | Diversity: 9.24 
Cost: 9290 | 